### Notebook with evaluation of predictive performance

This notebook compares the predictive performance which I obtain with different classification algorithms when using directly aggregated MD characteristics and features which were obtained from the per residue characteristics which were extracted at CeMM from the MD simulation.

Target values are:
1) The transmembran status of thje binding partner
2) A sequence derived evolutionary conservation index which is based on a 0.75 quantil summary of the sequence derived per residue descriptors whcih were provided by Evandro.  

### Extracted features

* MD_MMGBSA_..._mean: Average energy contribution (VdW, Elec, Polar, etc.) per interface residue. 

* MD_RMSF_mean: Average mean square fluctuation per interface

* Coords_RMSD_from_Centroid_CA: Measures the spread of the interface; calculated as the root mean square distance of CA atoms from their geometric center. (per interface)

### Fractions per interface: (count / divided by the interface length):

* Prop_ResCode_[rescode]: The fraction of specific residues at the interface

* Prop_AminoAcidType_[AminoacidType]: The fraction of specific types at the interface

* Prop_MD_[SecondaryStructure] (He/Be/Co): The fraction of secondary structure elements: He (Helices), Be (Beta-sheets), or Co (Coils and Loops).

* Prop_[contact_type] (hb, sb, vdw, etc.): The density of specific interaction types (e.g., Hydrogen Bonds, Salt Bridges) relative to the interface length.


### Target Labels:

* Transmembrane: Label True or false if Protein B is Transmembrane or not

* GMM_Label_Combined: Evolutionary label 0/1

* A description of the labels and features are detailed in data_overview.ipynb



In [1]:
# set the basepath to "one up"
basepath="../"
## set the path
import sys
sys.path.append(basepath+"course.lib/")
import ml_lib as mlib
## We use mapper.extract4ml to load the data.
## the default mode uses the transdoremd data as inpuit matrix X.
## The function allows us to specify several column selectors to assess the 
## information content in different feature sets.
## 1) column names for different processing steps
## mlib.col4tmlab="Transmembrane"                      ## column name in classic_raw holding the transmembran labels                                          
## mlib.col4eratelab="GMM_Label_Combined"              ## column name in classic_raw holding the evolutionary rate labels  
##
## input label columns to be used in function extract4ml as parameters Xcols with discardXcols=True
## mlib.rem_mns_fnam="../aggregation_tabular/colnams_remove_means.txt"      ## column names containing mean values
## mlib.rem_sums_fnam="../aggregation_tabular/colnams_remove_sums.txt"      ## column names containing sum values
## input label columns to be used in function extract4ml as parameters Xcols with discardXcols=True
## mlib.keep_but_small_counts="../aggregation_tabular/colnamesdropsmallcounts.txt" ## retains all but small counts 
## mlib.keep_mean_props="../aggregation_tabular/collnamesselmeanprops.txt"         ## retains mean values and proportions
## mlib.keep_sum_props="../aggregation_tabular/colnamesselsumprops.txt"            ## retains sum values and proportions (including all counts)
## to load a desired column set use function mlib.getcolnams() and provide the filename as argument.
## The columm names should be provided to function mlib.extract4ml together with an appropriately set flag discardXcols which
## controls whether the columns should be taken or removed.
## start with all data:
(X, y, ftrnams, sampleids)=mlib.extract4ml()

### Normalization with standard scaler

In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler as SSC
## we transform the data as a whole
Xtr=SSC().fit_transform(X)
## convert the labels
from sklearn.preprocessing import LabelEncoder as LBE
ytr=LBE().fit_transform(y)
## we work now with Xtr and ytr

## Accuracy and ROC curves with several sklearn classifiers
We use the
* Support vector classifier
* Multi Layer Perceptron Classifier
* Gaussian process classifier
* Random forest

## Evaluation methods

* Determine unbiased predictions with nested 10 folds cross valudation
* Calcuate sensitivity and specificity (or 1-specificity as false positive rate) for ROC curve construction
* Calculate the area under the ROC curve
* Estimate generalisation accuracy
* Estimate a modified Shannon channel capacity (estimated as average Kullback Leibler (KL) divergence between class prior and posterior) which sets the KL for misclassified samüles to zero 
* Compare the performace of all three methods with a McNemar test against the mlib.DefClassifier (majority vote also from ../course.lib)

The ROC curve coordinates and all evaluation metrics are calculated by function mlib.roc_auc_shannon_acc_mcnemar(P_class, y_true) which resides in ../course.lib/ml_lib.py


In [3]:
## set the path
import sys
sys.path.append(basepath+"course.lib/")
import ml_lib as mlib
import importlib
importlib.reload(mlib)
## import the sklearn model selection and assessment tools
from sklearn.model_selection import cross_val_predict as cvp
from sklearn.model_selection import StratifiedKFold as SKF
from sklearn.model_selection import GridSearchCV as GSCV

nfold=10
## create two SKF instances for grid search cv and cvp
skfo=SKF(nfold, shuffle=True)
skfi=SKF(nfold, shuffle=True)

## use skfo and cvp to create defaut predictions.
y_def=cvp(mlib.DefClassifier(), Xtr, ytr, cv=skfo)
acc_def=100.0*np.sum(y_def==ytr)/ytr.shape[0]

## start with the svc:
print("SVC")
from sklearn.svm import SVC
## hyperparameters of SVC: C=1.0, gamma="auto", "scale" and float values
minC=10**-6
maxC=50
Cvals=11
mingam=10**-2
maxgam=10
gamvals=11
## hyperparameters for rbf kernel based SVC
svcgrid={"C":np.linspace(minC, maxC, Cvals),
         "gamma":["auto", "scale"]+np.linspace(mingam, maxgam, gamvals).tolist()}
## define the SVC with rbf lernes alnd such that probabilities get estimated.
gssvc=GSCV(SVC(probability=True), svcgrid, n_jobs=-1, cv=skfi)
## unbiased predictions by nested cross validation using mlib.crossvalprobs
P_svc=mlib.crossvalprobs(gssvc, Xtr, ytr, cv=skfo)
rocx_svc, rocy_svc, auc_svc, shannon_svc, acc_svc, pval_svc=mlib.roc_auc_shannon_acc_mcnemar(P_svc[:,1], ytr)
rocstore=mlib.clssmetrics_2_store("SVC", rocx_svc, rocy_svc, auc_svc, acc_svc, shannon_svc, pval_svc)

SVC
round 1 of 10
round 2 of 10
round 3 of 10
round 4 of 10
round 5 of 10
round 6 of 10
round 7 of 10
round 8 of 10
round 9 of 10
round 10 of 10


In [4]:
## MLPC
print("MLPC")
from sklearn.neural_network import MLPClassifier as MLPC
mlpc_grid={"hidden_layer_sizes":[(12,), (25, ), (50,), (75,), (100,), (10, 10,)],
           "alpha":np.logspace(-4,2,7).tolist()}
## set a large maximum number of iterations to avoud convergence warnings.
maxit=2*10**5
mlpcinit=MLPC(activation="tanh", solver="lbfgs", max_iter=maxit)
## define the mlpc grid search
gsmlpc=GSCV(mlpcinit, mlpc_grid, n_jobs=-1, cv=skfi)
## unbiased predictions by nested cross validation using mlib.crossvalprobs
P_mlpc=mlib.crossvalprobs(gsmlpc, Xtr, ytr, cv=skfo)
rocx_mlpc, rocy_mlpc, auc_mlpc, shannon_mlpc, acc_mlpc, pval_mlpc=mlib.roc_auc_shannon_acc_mcnemar(P_mlpc[:,1], ytr)
rocstore=mlib.clssmetrics_2_store("MLPC", rocx_mlpc, rocy_mlpc, auc_mlpc, acc_mlpc, shannon_mlpc, pval_mlpc, rocstore)

MLPC
round 1 of 10
round 2 of 10


/home/psykacek/anaconda3/envs/ray/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


round 3 of 10


/home/psykacek/anaconda3/envs/ray/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


round 4 of 10
round 5 of 10


...

round 7 of 10


/home/psykacek/anaconda3/envs/ray/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


round 8 of 10
round 9 of 10
round 10 of 10


/home/psykacek/anaconda3/envs/ray/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


2


In [5]:
## GPC
print("GPC")
from sklearn.gaussian_process import GaussianProcessClassifier as GPC
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel
minval=10.0**-25
maxval=10.0**25
gpc_grid={"kernel":[RBF(length_scale_bounds=(minval, maxval)),                  ## RBF
                    Matern(length_scale_bounds=(minval, maxval), nu=0.5),       ## Matern with 3 nu params
                    Matern(length_scale_bounds=(minval, maxval), nu=1.5),
                    Matern(length_scale_bounds=(minval, maxval), nu=2.5)]}
gpcinit=GPC()
## define the gpc grid search
gsgpc=GSCV(gpcinit, gpc_grid, n_jobs=-1, cv=skfi)
## unbiased predictions by nested cross validation using mlib.crossvalprobs
P_gpc=mlib.crossvalprobs(gsgpc, Xtr, ytr, cv=skfo)
rocx_gpc, rocy_gpc, auc_gpc, shannon_gpc, acc_gpc, pval_gpc=mlib.roc_auc_shannon_acc_mcnemar(P_gpc[:,1], ytr)
rocstore=mlib.clssmetrics_2_store("GPC", rocx_gpc, rocy_gpc, auc_gpc, acc_gpc, shannon_gpc, pval_gpc, rocstore)

GPC
round 1 of 10
round 2 of 10
round 3 of 10
round 4 of 10
round 5 of 10
round 6 of 10
round 7 of 10
round 8 of 10
round 9 of 10
round 10 of 10


In [6]:
## Random forest
print("RFC")
from sklearn.ensemble import RandomForestClassifier as RFC
rfc_grid={"n_estimators":[100, 200],
          "criterion":["gini", "log_loss"],
          "max_depth":[5, 7, 9]
         }
rfcinit=RFC()
## define the gpc grid search
gsrfc=GSCV(rfcinit, rfc_grid, n_jobs=-1, cv=skfi)
## unbiased predictions by nested cross validation using mlib.crossvalprobs
P_rfc=mlib.crossvalprobs(gsrfc, Xtr, ytr, cv=skfo)
rocx_rfc, rocy_rfc, auc_rfc, shannon_rfc, acc_rfc, pval_rfc=mlib.roc_auc_shannon_acc_mcnemar(P_rfc[:,1], ytr)
rocstore=mlib.clssmetrics_2_store("RFC", rocx_rfc, rocy_rfc, auc_rfc, acc_rfc, shannon_rfc, pval_rfc, rocstore)

RFC
round 1 of 10
round 2 of 10
round 3 of 10
round 4 of 10
round 5 of 10
round 6 of 10
round 7 of 10
round 8 of 10
round 9 of 10
round 10 of 10


In [7]:
## add the data from L. Allmesberger (GNN results)
import pandas as pd
gnn_res_fnams=["../aggregation_tabular/GNN_max_ALL_FOLDS.csv", "../aggregation_tabular/GNN_mean_ALL_FOLDS.csv",
               "../aggregation_tabular/GNN_set2set_ALL_FOLDS.csv", "../aggregation_tabular/GNN_sum_ALL_FOLDS.csv"]
gnn_mthd_accros=["GNNX", "GNNM", "GNNS2S", "GNNS"]
p_col="y_prob"
y_col="y_true"
for mid, fnam in enumerate(gnn_res_fnams):
    ## read and process
    dfrm=pd.read_csv(fnam)
    Probs=dfrm.loc[:, p_col].to_numpy()
    y_true=dfrm.loc[:, y_col].to_numpy()
    rocx_gnn, rocy_gnn, auc_gnn, shannon_gnn, acc_gnn, pval_gnn=mlib.roc_auc_shannon_acc_mcnemar(Probs, y_true)
    rocstore=mlib.clssmetrics_2_store(gnn_mthd_accros[mid], rocx_gnn, rocy_gnn, auc_gnn, acc_gnn, shannon_gnn, pval_gnn, rocstore)
## store the information in the default location (mlib.defaultstore="../results/classmetrics.pkl")
mlib.dumpstore(rocstore)

/home/psykacek/sciwrk/proposalcemm/project/ghdir/notebooks/../course.lib/ml_lib.py:166: RuntimeWarning: divide by zero encountered in log2
  Plg=np.log2(P1)-np.log2(P2)
